In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix



In [ ]:
FIG = Path("../figures")
FIG.mkdir(parents=True, exist_ok=True)
RANDOM_SEED = 42



In [ ]:
# ======================
# 1. СИНТЕТИЧЕСКИЕ ДАННЫЕ (Santander-like)
# ======================
X, y = make_classification(
    n_samples=8000,
    n_features=100,
    n_informative=15,
    n_redundant=20,
    n_repeated=0,
    weights=[0.9, 0.1],
    random_state=RANDOM_SEED,
)
feature_names = [f"var_{i}" for i in range(X.shape[1])]
df = pd.DataFrame(X, columns=feature_names)
df["target"] = y



In [ ]:
print("Number of samples:", df.shape[0])
print("Number of features:", X.shape[1])
print("Positive rate:", round(y.mean(), 4))



In [ ]:
df.head()



In [ ]:
df.describe().T.head(10)



In [ ]:
# Баланс классов (типичный дисбаланс как в Santander)
class_counts = pd.Series(y).value_counts().sort_index()

plt.figure(figsize=(7, 4))
sns.barplot(
    x=["negative", "positive"],
    y=class_counts.values,
    hue=["negative", "positive"],
    palette="viridis",
    legend=False,
)
plt.title("Target class balance", fontsize=14, pad=12)
plt.ylabel("Count")
plt.grid(axis="y", linestyle="--", alpha=0.7)
for i, c in enumerate(class_counts.values):
    plt.text(i, c + 20, str(c), ha="center")
plt.tight_layout()
plt.show()



In [ ]:
# Распределения нескольких признаков
cols = feature_names[:6]
fig, axes = plt.subplots(2, 3, figsize=(12, 7))
for ax, col in zip(axes.flatten(), cols):
    sns.histplot(df[col], bins=30, ax=ax, color="#4c72b0", edgecolor="black")
    ax.set_title(col)
plt.suptitle("Feature distributions (first 6)", fontsize=14)
plt.tight_layout()
plt.show()



In [ ]:
# Корреляция подмножества признаков
sub = feature_names[:20] + ["target"]
corr = df[sub].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr, cmap="coolwarm", center=0, square=False)
plt.title("Correlation (first 20 features + target)", fontsize=13, pad=10)
plt.tight_layout()
plt.show()



In [ ]:
# ======================
# 2. TRAIN / TEST SPLIT
# ======================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_SEED
)
print("Train:", X_train.shape, "Test:", X_test.shape)



In [ ]:
# ======================
# 3. PIPELINE: SelectKBest + LogReg
# ======================
pipe_lr = Pipeline([
    ("sel", SelectKBest(f_classif, k=30)),
    ("sc", StandardScaler()),
    ("m", LogisticRegression(max_iter=2000, random_state=RANDOM_SEED)),
])
cv_lr = cross_val_score(pipe_lr, X_train, y_train, cv=3, scoring="roc_auc")
print("logreg_kbest CV AUC:", round(cv_lr.mean(), 4), "+/-", round(cv_lr.std(), 4))



In [ ]:
# ======================
# 4. HISTGRADIENTBOOSTING
# ======================
hgb = HistGradientBoostingClassifier(
    max_depth=6,
    learning_rate=0.08,
    random_state=RANDOM_SEED,
)
cv_hgb = cross_val_score(hgb, X_train, y_train, cv=3, scoring="roc_auc")
print("hgb CV AUC:", round(cv_hgb.mean(), 4), "+/-", round(cv_hgb.std(), 4))



In [ ]:
# Обучение на полном train и оценка на test
pipe_lr.fit(X_train, y_train)
hgb.fit(X_train, y_train)

proba_lr = pipe_lr.predict_proba(X_test)[:, 1]
proba_hgb = hgb.predict_proba(X_test)[:, 1]

print("logreg_kbest test AUC:", round(roc_auc_score(y_test, proba_lr), 4))
print("hgb test AUC:", round(roc_auc_score(y_test, proba_hgb), 4))



In [ ]:
print(classification_report(y_test, (proba_hgb >= 0.5).astype(int), digits=3))



In [ ]:
# Сравнение AUC
scores = pd.Series({
    "LogReg+KBest": roc_auc_score(y_test, proba_lr),
    "HistGBM": roc_auc_score(y_test, proba_hgb),
})

plt.figure(figsize=(7, 4))
sns.barplot(x=scores.index, y=scores.values, palette="viridis")
plt.ylim(0.5, 1.0)
plt.ylabel("ROC-AUC")
plt.title("Model comparison (test AUC)", fontsize=14, pad=12)
for i, v in enumerate(scores.values):
    plt.text(i, v + 0.01, f"{v:.3f}", ha="center")
plt.tight_layout()
plt.show()



In [ ]:
# Матрица ошибок HistGBM
cm = confusion_matrix(y_test, (proba_hgb >= 0.5).astype(int))
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix (HistGBM @ 0.5)")
plt.tight_layout()
plt.show()



In [ ]:
# Какие признаки выбрал SelectKBest
selected = pipe_lr.named_steps["sel"].get_support()
selected_names = [n for n, s in zip(feature_names, selected) if s]
print("Selected K features:", len(selected_names))
print(selected_names[:15], "...")

